# Notebook 09 - Market Segmentation & Expansion Strategy
# Convert opportunity scores into business actions


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Upload scored market data

from google.colab import files

uploaded = files.upload()

market_scores = pd.read_csv(
    "market_opportunity_scores.csv"
)

locations = pd.read_csv(
    "location_opportunity_features.csv"
)

print("Markets:", market_scores.shape)
print("Locations:", locations.shape)

Saving market_opportunity_scores.csv to market_opportunity_scores (1).csv
Saving location_opportunity_features.csv to location_opportunity_features (1).csv
Markets: (48, 14)
Locations: (3232, 36)


In [ ]:
# Check market score ranges

print(
    market_scores["Opportunity_Score"]
    .describe()
    .round(2)
)

count    48.00
mean     58.53
std      10.54
min      37.52
25%      50.39
50%      59.54
75%      66.18
max      74.99
Name: Opportunity_Score, dtype: float64


In [ ]:
# Create data-driven score thresholds

q25 = market_scores["Opportunity_Score"].quantile(0.25)
q50 = market_scores["Opportunity_Score"].quantile(0.50)
q75 = market_scores["Opportunity_Score"].quantile(0.75)

print("25th percentile:", round(q25, 2))
print("Median:", round(q50, 2))
print("75th percentile:", round(q75, 2))

25th percentile: 50.39
Median: 59.54
75th percentile: 66.18


In [ ]:
# Segment markets into strategic actions

def classify_market(row):

    score = row["Opportunity_Score"]
    risk = row["Competition_Risk"]

    if score >= q75 and risk < 0.50:
        return "EXPAND"

    elif score >= q75 and risk >= 0.50:
        return "BATTLEGROUND"

    elif score >= q50:
        return "WATCH"

    else:
        return "LOW PRIORITY"


market_scores["Strategy"] = market_scores.apply(
    classify_market,
    axis=1
)

print(
    market_scores["Strategy"]
    .value_counts()
)

Strategy
LOW PRIORITY    24
EXPAND          12
WATCH           12
Name: count, dtype: int64


In [ ]:
# View market recommendations

strategy_table = (
    market_scores[
        [
            "assigned_market",
            "Opportunity_Score",
            "Demand_Strength",
            "Competition_Risk",
            "Rank_Stability",
            "Strategy"
        ]
    ]
    .sort_values(
        "Opportunity_Score",
        ascending=False
    )
)

strategy_table.head(20)

,assigned_market,Opportunity_Score,Demand_Strength,Competition_Risk,Rank_Stability,Strategy
0,coimbatore,74.992844,0.908019,0.104709,1.732051,EXPAND
1,vellore,74.142026,0.785377,0.027159,1.732051,EXPAND
2,madurai,73.992670,0.841981,0.077736,1.000000,EXPAND
3,palakkad,73.530663,0.735849,0.015970,3.511885,EXPAND
4,nagpur,73.052309,0.903302,0.175774,2.516611,EXPAND
5,surat,72.307090,0.903302,0.128353,2.516611,EXPAND
6,nashik,70.628958,0.882075,0.158972,2.081666,EXPAND
7,davanagere,70.617887,0.551887,0.014259,6.557439,EXPAND
8,pune,68.763810,0.945755,0.279043,4.163332,EXPAND
9,kolkata,67.843138,0.905660,0.311577,4.358899,EXPAND


In [ ]:
import plotly.express as px

# Top 20 markets for readable executive view
plot_df = (
    market_scores
    .sort_values(
        ["Opportunity_Score", "Rank_Stability"],
        ascending=[False, True]
    )
    .head(20)
    .sort_values("Opportunity_Score")
    .copy()
)

plot_df["Market"] = (
    plot_df["assigned_market"]
    .str.title()
)

strategy_colors = {
    "EXPAND": "#2A9D8F",
    "WATCH": "#E9C46A",
    "LOW PRIORITY": "#B8B8B8"
}

fig = px.bar(
    plot_df,

    x="Opportunity_Score",
    y="Market",

    orientation="h",

    color="Strategy",
    color_discrete_map=strategy_colors,

    text="Opportunity_Score",

    hover_data={
        "Demand_Strength": ":.2f",
        "Competition_Risk": ":.2f",
        "Rank_Stability": ":.2f",
        "Strategy": True
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",

    marker_line_width=0,

    hovertemplate=(
        "<b>%{y}</b><br>"
        "Opportunity Score: %{x:.1f}<br>"
        "Strategy: %{customdata[3]}<br>"
        "Demand Strength: %{customdata[0]:.2f}<br>"
        "Competition Risk: %{customdata[1]:.2f}<br>"
        "Rank Stability: %{customdata[2]:.2f}"
        "<extra></extra>"
    )
)

fig.update_layout(

    title=dict(
        text=(
            "<b>MARKET EXPANSION PRIORITY</b><br>"
            "<span style='font-size:13px;color:#777777'>"
            "Top markets ranked by opportunity strength and strategic action"
            "</span>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(
            size=22,
            color="#263B5E"
        )
    ),

    xaxis=dict(
        title="Location Opportunity Score →",
        range=[
            max(0, plot_df["Opportunity_Score"].min() - 5),
            plot_df["Opportunity_Score"].max() + 6
        ],
        gridcolor="#EEEEEE",
        zeroline=False
    ),

    yaxis=dict(
        title="",
        showgrid=False,
        tickfont=dict(size=12)
    ),

    legend=dict(
        title="Recommended Action",
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.06
    ),

    bargap=0.28,

    plot_bgcolor="#FCFCFC",
    paper_bgcolor="white",

    width=1000,
    height=720,

    margin=dict(
        t=125,
        l=145,
        r=80,
        b=75
    )
)

fig.show()

In [ ]:
# Top expansion recommendations

expansion_shortlist = (
    market_scores[
        market_scores["Strategy"] == "EXPAND"
    ]
    .sort_values(
        [
            "Opportunity_Score",
            "Rank_Stability"
        ],
        ascending=[False, True]
    )
)

expansion_shortlist[
    [
        "assigned_market",
        "Opportunity_Score",
        "Demand_Strength",
        "Competition_Risk",
        "Rank_Stability"
    ]
]

,assigned_market,Opportunity_Score,Demand_Strength,Competition_Risk,Rank_Stability
0,coimbatore,74.992844,0.908019,0.104709,1.732051
1,vellore,74.142026,0.785377,0.027159,1.732051
2,madurai,73.992670,0.841981,0.077736,1.000000
3,palakkad,73.530663,0.735849,0.015970,3.511885
4,nagpur,73.052309,0.903302,0.175774,2.516611
5,surat,72.307090,0.903302,0.128353,2.516611
6,nashik,70.628958,0.882075,0.158972,2.081666
7,davanagere,70.617887,0.551887,0.014259,6.557439
8,pune,68.763810,0.945755,0.279043,4.163332
9,kolkata,67.843138,0.905660,0.311577,4.358899


In [ ]:
# Final strategic priority table

priority_table = (
    market_scores[
        [
            "assigned_market",
            "Opportunity_Score",
            "Demand_Strength",
            "Competition_Risk",
            "Rank_Stability",
            "Strategy"
        ]
    ]
    .sort_values(
        ["Opportunity_Score", "Rank_Stability"],
        ascending=[False, True]
    )
    .copy()
)

priority_table["Priority_Rank"] = range(
    1,
    len(priority_table) + 1
)

priority_table[
    [
        "Priority_Rank",
        "assigned_market",
        "Opportunity_Score",
        "Strategy",
        "Rank_Stability"
    ]
].head(15)

,Priority_Rank,assigned_market,Opportunity_Score,Strategy,Rank_Stability
0,1,coimbatore,74.992844,EXPAND,1.732051
1,2,vellore,74.142026,EXPAND,1.732051
2,3,madurai,73.992670,EXPAND,1.000000
3,4,palakkad,73.530663,EXPAND,3.511885
4,5,nagpur,73.052309,EXPAND,2.516611
5,6,surat,72.307090,EXPAND,2.516611
6,7,nashik,70.628958,EXPAND,2.081666
7,8,davanagere,70.617887,EXPAND,6.557439
8,9,pune,68.763810,EXPAND,4.163332
9,10,kolkata,67.843138,EXPAND,4.358899


In [ ]:
# Strategic market summary

strategy_summary = (
    market_scores
    .groupby("Strategy", as_index=False)
    .agg(
        Markets=("assigned_market", "count"),
        Avg_Opportunity=("Opportunity_Score", "mean"),
        Avg_Demand=("Demand_Strength", "mean"),
        Avg_Competition=("Competition_Risk", "mean")
    )
    .round(2)
)

strategy_summary

,Strategy,Markets,Avg_Opportunity,Avg_Demand,Avg_Competition
0,EXPAND,12,71.11,0.83,0.14
1,LOW PRIORITY,24,49.49,0.28,0.15
2,WATCH,12,64.04,0.65,0.15


In [ ]:
# Save Notebook 9 outputs

market_scores.to_csv(
    "market_expansion_strategy.csv",
    index=False
)

priority_table.to_csv(
    "market_priority_ranking.csv",
    index=False
)

expansion_shortlist.to_csv(
    "expansion_shortlist.csv",
    index=False
)

print("Notebook 9 outputs saved.")

Notebook 9 outputs saved.


In [ ]:
# Download Notebook 9 outputs

from google.colab import files

files.download("market_expansion_strategy.csv")
files.download("market_priority_ranking.csv")
files.download("expansion_shortlist.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>